# Online model serving via `ml_app_client`

This notebook creates a stable model service, performs an auditable prediction, reads its Inference Log, and shows how to queue challenger replay. The selected model must already be in the `production` stage.

In [1]:
from ml_app_client import MLAppClient

# Uses ML_APP_ACCESS_TOKEN when configured; otherwise asks for a normal user's credentials.
client = MLAppClient.connect()

In [ ]:
updated_model = client.promote_model(
    "Estates Sell Prices - AutoML - Model",
    "developed",
    version=3,  # Optional: select a specific version.
)

updated_model["version"], updated_model["stage"]

('v3', 'developed')

In [2]:
versions = ['v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7']
updated_models = client.promote_model_versions(
    "Storage Subscription Churn - Experiment 3- AutoML champion",
    "archived",
    versions=versions,
)
[(model['version'], model['stage']) for model in updated_models]

[('v1', 'archived'),
 ('v2', 'archived'),
 ('v3', 'archived'),
 ('v4', 'archived'),
 ('v5', 'archived'),
 ('v6', 'archived'),
 ('v7', 'archived')]

In [ ]:
service = client.create_deployment(
    name="Estates Sell Prices Service",
    model_name="Estates Sell Prices - AutoFEML - Model",
)
service.endpoint_url

In [ ]:
result = client.predict(
    service,
    record_id="estate-2026-0001",
    features={"area": 84.0, "rooms": 4, "year_built": 2018},
    idempotency_key="valuation-estate-2026-0001",
)
result

In [ ]:
history = client.inference_history(service, record_id="estate-2026-0001")
history["items"]

In [ ]:
# After assigning a staging/production model as challenger in a new revision:
# replay = client.replay_challenger(
#     service, challenger_model_id="MODEL_VERSION_ID", max_requests=1000
# )